# IICA Forth — Language as Lattice

This notebook explores the **IICA Forth** language where:
- Every word definition IS an HLLSet
- The dictionary IS a lattice of content-addressed meanings
- BSS similarity searches the semantic space

**Kernel:** Python 3. Run cells top-to-bottom.

---
## Setup
Load the IICA dictionary engine.

In [1]:
import json, os, subprocess, sys
from pathlib import Path

HLLSET = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/target/debug/hllset"

def hllset(script):
    proc = subprocess.run([HLLSET, "-e", script], capture_output=True, text=True, timeout=30)
    if proc.returncode != 0: raise RuntimeError(proc.stderr.strip())
    return json.loads(proc.stdout.strip())

def inscribe(tokens):
    j = "{" + ", ".join(f'"{t}"' for t in tokens) + "}"
    script = f"local e = hllset.inscribe({j}); local k = e:key(); return {{key=k, card=#e}}"
    return hllset(script)

def bss_inscribe(ta, tb):
    ja = "{" + ", ".join(f'"{t}"' for t in ta) + "}"
    jb = "{" + ", ".join(f'"{t}"' for t in tb) + "}"
    script = f'local a=hllset.inscribe({ja}); local b=hllset.inscribe({jb}); return a:bss_inclusion(b)'
    return hllset(script)

class IICADict:
    def __init__(self): self.words = {}
    def define(self, name, tokens):
        r = inscribe(tokens)
        self.words[name] = {"tokens": tokens, "key": r["key"], "card": r["card"]}
        return r["key"]
    def similar(self, a, b):
        return bss_inscribe(self.words[a]["tokens"], self.words[b]["tokens"])
    def search(self, query, top=5):
        scored = [(bss_inscribe(query, w["tokens"]), name, w["key"], w["card"]) for name, w in self.words.items()]
        scored.sort(key=lambda x: -x[0])
        return scored[:top]
    def projection(self, name):
        tokens = self.words[name]["tokens"]
        return {w: self.similar(name, w) for w in self.words if w != name}
    def __len__(self): return len(self.words)

d = IICADict()
print(f"IICADict ready. Define words with d.define(name, [tokens])")

IICADict ready. Define words with d.define(name, [tokens])


---
## Scenario 1: Identical Definitions = Identical HLLSet
Two words with the same token list produce the same content key. The dictionary
is self-deduplicating — you can't create two different words with the same body.

In [2]:
d.define("add",     ["union", "merge", "combine", "crdt"])
d.define("merge2",  ["union", "merge", "combine", "crdt"])  # same tokens
d.define("subtract",["diff", "remove", "exclude"])

print("add key:     ", d.words["add"]["key"])
print("merge2 key:  ", d.words["merge2"]["key"])
print("subtract key:", d.words["subtract"]["key"])
print()
same = d.words["add"]["key"] == d.words["merge2"]["key"]
print(f"add == merge2? {same}  ← IICA: same definition → same identity")

add key:      h:3baa4cce3404c6814a83fb75e87ceae252643326
merge2 key:   h:3baa4cce3404c6814a83fb75e87ceae252643326
subtract key: h:1b87b53bbb5a2bb94eb92dea03ce5d3f1a02bef6

add == merge2? True  ← IICA: same definition → same identity


---
## Scenario 2: BSS Similarity Between Words
BSSτ(A, B) = how much of B's definition is contained in A.
τ=1.0 means A fully contains B (or they're identical). τ=0 means no overlap.

In [3]:
pairs = [
    ("add", "merge2"),
    ("add", "subtract"),
    ("merge2", "subtract"),
]
for a, b in pairs:
    tau = d.similar(a, b)
    print(f"BSS({a:8s} → {b:8s}) = {tau:.3f}")

BSS(add      → merge2  ) = 1.000
BSS(add      → subtract) = 0.000
BSS(merge2   → subtract) = 0.000


---
## Scenario 3: Dictionary Growth — Semantic Clusters
Build a larger dictionary organized by semantic domains. Each word's
definition tokens encode its meaning. We can visualize BSS similarity
as a matrix.

In [4]:
# ── Mathematics domain ──
math_words = [
    ("vector",  ["direction", "magnitude", "space", "linear"]),
    ("matrix",  ["grid", "transform", "linear", "multiply", "rows", "columns"]),
    ("tensor",  ["multi-dim", "generalize", "transform", "matrix", "vector"]),
    ("scalar",  ["number", "magnitude", "single", "value"]),
    ("dot",     ["multiply", "inner", "product", "scalar", "vector"]),
    ("cross",   ["multiply", "outer", "product", "vector", "perpendicular"]),
]
for name, tokens in math_words:
    d.define(name, tokens)

# ── Computing domain ──
comp_words = [
    ("hash",    ["function", "map", "fingerprint", "deterministic"]),
    ("cache",   ["memory", "store", "fast", "temporary", "lookup"]),
    ("pipeline",["stage", "sequential", "flow", "throughput", "parallel"]),
    ("register",["store", "flip-flop", "bit", "state", "clock"]),
    ("fpga",    ["reconfigurable", "gate", "array", "hardware", "parallel"]),
]
for name, tokens in comp_words:
    d.define(name, tokens)

print(f"Dictionary size: {len(d)} words")
for name in sorted(d.words.keys()):
    w = d.words[name]
    print(f"  {name:10s} → {w["key"][:20]}... [{w["card"]}]")

Dictionary size: 14 words
  add        → h:3baa4cce3404c6814a... [4.0]
  cache      → h:0a835f859d71a98943... [5.0]
  cross      → h:a24243b21752ba9c72... [5.0]
  dot        → h:ef8abbeb4ad5b21a0a... [5.0]
  fpga       → h:7890ed6875df9bbaa8... [5.0]
  hash       → h:63b07811cb6e28aa27... [4.0]
  matrix     → h:1c821b0b66687f491e... [6.0]
  merge2     → h:3baa4cce3404c6814a... [4.0]
  pipeline   → h:4785d792747a084256... [5.0]
  register   → h:f36f2cd219d8e09471... [5.0]
  scalar     → h:3e6ac5526b6b6758f7... [4.0]
  subtract   → h:1b87b53bbb5a2bb94e... [3.0]
  tensor     → h:a1e7647eb2c601256c... [5.0]
  vector     → h:ec39a484a06e002e58... [4.0]


---
## Scenario 4: Content-Based Search
Find words by describing what you want — not by name, but by meaning.
The query tokens are inscribed as an HLLSet, then BSS-ranked against all words.

In [5]:
queries = [
    ["linear", "transform", "space"],
    ["memory", "cache", "fast"],
    ["parallel", "hardware", "gate"],
    ["multiply", "inner", "scalar"],
    ["deterministic", "fingerprint", "function"],
]

for q in queries:
    print(f"query: [{", ".join(q)}]")
    for tau, name, key, card in d.search(q, 3):
        bar = "█" * int(tau * 20)
        print(f"  {tau:.3f} {bar:20s} {name:10s}  (card={card})")
    print()

query: [linear, transform, space]
  0.500 ██████████           vector      (card=4.0)
  0.333 ██████               matrix      (card=6.0)
  0.200 ████                 tensor      (card=5.0)

query: [memory, cache, fast]
  0.400 ████████             cache       (card=5.0)
  0.000                      add         (card=4.0)
  0.000                      merge2      (card=4.0)

query: [parallel, hardware, gate]
  0.600 ████████████         fpga        (card=5.0)
  0.200 ████                 pipeline    (card=5.0)
  0.000                      add         (card=4.0)

query: [multiply, inner, scalar]
  0.600 ████████████         dot         (card=5.0)
  0.200 ████                 cross       (card=5.0)
  0.167 ███                  matrix      (card=6.0)

query: [deterministic, fingerprint, function]
  0.750 ███████████████      hash        (card=4.0)
  0.000                      add         (card=4.0)
  0.000                      merge2      (card=4.0)



---
## Scenario 5: Word Projection
For a given word, show its BSS similarity to every other word.
This is the word's "meaning vector" in the dictionary lattice.

In [6]:
def project(d, word_name):
    "Show BSS projection of a word against all other words in the dictionary."
    scores = [(d.similar(word_name, r), r) for r in d.words.keys() if r != word_name]
    scores.sort(key=lambda x: -x[0])
    for tau, name in scores:
        bar = "█" * int(tau * 20)
        print(f"  {tau:.3f} {bar:20s} {name}")

print("── tensor projection ──")
project(d, "tensor")
print()
print("── fpga projection ──")
project(d, "fpga")

── tensor projection ──
  0.200 ████                 dot
  0.200 ████                 cross
  0.167 ███                  matrix
  0.000                      add
  0.000                      merge2
  0.000                      subtract
  0.000                      vector
  0.000                      scalar
  0.000                      hash
  0.000                      cache
  0.000                      pipeline
  0.000                      register
  0.000                      fpga

── fpga projection ──
  0.200 ████                 pipeline
  0.000                      add
  0.000                      merge2
  0.000                      subtract
  0.000                      vector
  0.000                      matrix
  0.000                      tensor
  0.000                      scalar
  0.000                      dot
  0.000                      cross
  0.000                      hash
  0.000                      cache
  0.000                      register


---
## Scenario 6: Cross-Domain Semantic Distance
How far apart are words from different semantic domains?
Low BSS means well-separated concepts; high BSS means semantic overlap.

In [7]:
math_w = ["vector", "matrix", "tensor", "scalar", "dot", "cross"]
comp_w = ["hash", "cache", "pipeline", "fpga", "vector"]

print(f'{"":10s}', end="")
for cw in comp_w:
    print(f"{cw:>10s}", end="")
print()

for mw in math_w:
    print(f"{mw:10s}", end="")
    for cw in comp_w:
        tau = d.similar(mw, cw)
        print(f"{tau:>10.3f}", end="")
    print()

print()
print("Math words vs Computing words — low overlap = clean separation")

                hash     cache  pipeline      fpga    vector
vector         0.000     0.000     0.000     0.000     1.000
matrix         0.000     0.000     0.000     0.000     0.250
tensor         0.000     0.000     0.000     0.000     0.000
scalar         0.000     0.000     0.000     0.000     0.250
dot            0.000     0.000     0.000     0.000     0.000
cross          0.000     0.000     0.000     0.000     0.000

Math words vs Computing words — low overlap = clean separation


---
## Summary

| Scenario | What it demonstrates |
|----------|---------------------|
| **1. Identity** | Same tokens → same HLLSet key (IICA: Immutable, Idempotent) |
| **2. BSS** | τ=1.0 for identical, τ=0.0 for disjoint |
| **3. Growth** | Dictionary scales with O(1) comparisons |
| **4. Search** | Query by meaning, not by name — BSS-ranked retrieval |
| **5. Projection** | Each word's "meaning vector" in the dictionary lattice |
| **6. Cross-domain** | Math vs Computing — clean semantic separation |

The IICA Forth language is **self-introspecting**: words can search for each other,
compare definitions, and find semantic neighbors. The dictionary IS the program,
and the program IS an HLLSet.